# agentflowkit — Quick Start

This notebook walks you through the core features of `agentflowkit` in under 5 minutes.

**Prerequisites:**
```bash
pip install agentflowkit
```
You also need a Groq API key (free at [console.groq.com](https://console.groq.com)) or an OpenAI API key.

In [ ]:
import os

# Set your API key here (or export it in your shell before launching Jupyter)
# os.environ["GROQ_API_KEY"] = "gsk_..."
# os.environ["OPENAI_API_KEY"] = "sk-..."

## 1. Configure the LLM

`LLM` wraps any OpenAI-compatible provider. Here we use Groq (fast, free tier available).

In [ ]:
from agentflow import LLM

# Groq — free tier, very fast
llm = LLM(
    model="llama-3.3-70b-versatile",
    base_url="https://api.groq.com/openai/v1",
    api_key=os.environ.get("GROQ_API_KEY"),
)

# Or OpenAI:
# llm = LLM(model="gpt-4o-mini", api_key=os.environ.get("OPENAI_API_KEY"))

print(f"Using model: {llm.model}")

## 2. Define Agents

Use the `@Agent` decorator on any async function. The function receives `task` and `context` and returns the **user message** to send to the LLM.

In [ ]:
from agentflow import Agent


@Agent(name="researcher", role="Research Analyst")
async def researcher(task: str, context: dict) -> str:
    return f"Research this topic and provide key insights: {task}"

@Agent(name="fact_checker", role="Fact Checker")
async def fact_checker(task: str, context: dict) -> str:
    return f"Find 5 specific statistics or facts about: {task}"

@Agent(name="writer", role="Content Writer")
async def writer(task: str, context: dict) -> str:
    research = context.get("researcher", "")
    facts = context.get("fact_checker", "")
    return (
        f"Write a concise 3-paragraph article about: {task}\n\n"
        f"Use this research:\n{research[:300]}\n\n"
        f"And these facts:\n{facts[:300]}"
    )

print("Agents defined:", researcher, fact_checker, writer)

## 3. Build a Pipeline

Add agents with optional `depends_on`. Agents **without** shared dependencies run **in parallel**.

In [ ]:
from agentflow import Pipeline

pipe = Pipeline(llm=llm)

# researcher and fact_checker have no dependencies → they run in PARALLEL (Level 0)
pipe.add(researcher)
pipe.add(fact_checker)

# writer depends on both → runs after both complete (Level 1)
pipe.add(writer, depends_on=["researcher", "fact_checker"])

print("Pipeline configured with", len(pipe._nodes), "agents")

## 4. Run the Pipeline

In [ ]:
import time

TOPIC = "Quantum Computing applications in drug discovery"

start = time.perf_counter()
result = await pipe.run(TOPIC)
elapsed = time.perf_counter() - start

print(f"Completed in {elapsed:.2f}s")
print(f"Run ID: {result.run_id}")
print(f"Total tokens: {result.total_tokens}")
print(f"Levels executed: {result.levels_executed}")
print()
print("=" * 60)
print(result.output)

## 5. Access Individual Agent Results

In [ ]:
for name, agent_result in result.results.items():
    print(f"Agent: {name}")
    print(f"  Level:  {agent_result.level}")
    print(f"  Tokens: {agent_result.tokens_used}")
    print(f"  Time:   {agent_result.duration:.2f}s")
    print(f"  Cached: {agent_result.cached}")
    print(f"  Preview: {agent_result.output[:100]}...")
    print()

## 6. Stream Real-Time Events

In [ ]:
import ipywidgets as widgets
from IPython.display import display

out = widgets.Output()
display(out)

pipe2 = Pipeline(llm=llm)
pipe2.add(researcher)
pipe2.add(fact_checker)
pipe2.add(writer, depends_on=["researcher", "fact_checker"])

async for event in pipe2.stream("Gene therapy breakthroughs 2024"):
    with out:
        if event.type == "agent_start":
            print(f"▶ {event.agent} started (level {event.data.get('level', 0)})")
        elif event.type == "agent_complete":
            print(f"✓ {event.agent} — {event.data['tokens']} tokens in {event.data['duration']:.1f}s")
        elif event.type == "pipeline_complete":
            print(f"\nDone! {event.data['total_tokens']} total tokens · {event.data['levels_executed']} levels")

## 7. Add Caching to Avoid Repeated API Calls

In [ ]:
from agentflow import InMemoryCache

cache = InMemoryCache(default_ttl=3600)  # cache for 1 hour
cached_llm = LLM(
    model=llm.model,
    base_url="https://api.groq.com/openai/v1",
    api_key=os.environ.get("GROQ_API_KEY"),
    cache=cache,
)

cached_pipe = Pipeline(llm=cached_llm)
cached_pipe.add(researcher)

# First call — hits the API
t0 = time.perf_counter()
r1 = await cached_pipe.run(TOPIC)
t1 = time.perf_counter()

# Second call — hits the cache (nearly instant)
r2 = await cached_pipe.run(TOPIC)
t2 = time.perf_counter()

print(f"First call:  {t1-t0:.2f}s, cached={r1.results['researcher'].cached}")
print(f"Second call: {t2-t1:.2f}s, cached={r2.results['researcher'].cached}")
print(f"Speedup: {(t1-t0)/(t2-t1):.0f}x")